# Choosing and Coordinating Collections


```{contents}
:local:
:depth: 2
```


```{index} Coordinate Indexes with One Consistency Story
```



```{index} Coordinate Indexes with One Consistency Story
```

## Coordinate Indexes with One Consistency Story



A dispatch system wants quick lookup by ticket ID and repeated service by priority. One collection may not efficiently supply both operations under the required policies. A dictionary can locate the current ticket by ID while a priority queue selects service candidates. Using two indexes introduces an obligation: their views must not cause cancelled or superseded work to be served.

One design permits stale queue entries and checks them when removed. The dictionary holds authoritative current state. A queue candidate is served only if its ID still exists and its version matches the current ticket. Cancellation removes or marks the authoritative record; old queue entries can be skipped later. Versioning matters when the same ID can be updated: mere presence is not proof that the old priority entry still describes the current work.

| Event | Authoritative lookup | Old queue entry |
| --- | --- | --- |
| Register ticket | Present with current version | Eligible when selected |
| Cancel ticket | Absent or explicitly cancelled | Skip |
| Change priority/version | Updated current version | Skip if its version is stale |

The alternative is to remove/update queue entries eagerly, which needs suitable index support and its own maintenance rules. Compare complete workflows rather than declaring either approach universally faster. Lazy cleanup simplifies some updates but can retain stale storage and make later removals perform extra work.

Guided trace: register ticket A, cancel A, register a new version of A, then remove an old queue candidate. Explain why matching only the ID can serve the wrong version. Extend the trace with a failed lookup and check that it changes neither index. The existing cancellation example is an entry point to this consistency reasoning, not a guarantee that every multi-index update is automatically transactional.

For related explanation, see the further reading below.[^development-reading]


A service application may need exact lookup, uniqueness checks, ordered reports, and priority dispatch at the same time. Name each operation before choosing a representation. Multiple indexes are useful only if updates preserve a clear agreement between them.


```{index} Match the Structure to the Workload
```

## Match the Structure to the Workload




```{index} workload; data-structure choice
```

The same service desk has several access needs. Use a priority queue for the next urgent ticket, a dictionary for its owner, and a set for uniqueness or closed-ticket membership. A tree supports ordered keys and range queries. Each structure preserves a different relationship; the application must keep shared IDs consistent.

| Need | Structure | Cost under stated assumptions |
| --- | --- | --- |
| Select next urgency | Heap priority queue | O(log n) repair; backing-array growth can make one enqueue O(n). |
| Find owner by ID | Hash dictionary | Expected O(1) with well-distributed hashing and bounded load; worst-case O(n). |
| Reject duplicate ID | Hash set or dictionary | Expected O(1) under the same assumptions. |
| Ordered interval report | Balanced search tree | O(log n + k) for k results; the Chapter 16 unbalanced tree depends on actual height. |
| Process by arrival | FIFO queue | Constant-time endpoints in the linked representation. |

These bounds assume fixed-cost comparison and hashing. They describe operations, not measured milliseconds. A tiny list can still be a practical choice when there are few records and little repeated lookup.

Download {download}`the complete heap and chained-map source with verification <../../demos/19/HeapHashStructures.cs>` to use in a console project.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var urgentWork = new PriorityQueue<string, int>();
urgentWork.Enqueue("Weekly report", 5); urgentWork.Enqueue("Database outage", 0);
var ownerByTicket = new Dictionary<string, string> { ["T-100"] = "Amina" };
var closedTickets = new HashSet<string> { "T-099" };
Console.WriteLine($"next={urgentWork.Dequeue()}");
Console.WriteLine($"owner={ownerByTicket["T-100"]}");
Console.WriteLine($"closed={closedTickets.Contains("T-099")}");

next=Database outage
owner=Amina
closed=True


```{index} Count Work Before Timing
```

## Count Work Before Timing




```{index} hashing; operation count; performance measurement
```

One stopwatch measurement includes runtime warm-up, scheduling noise, and timer granularity. It cannot establish a complexity bound. First compare the amount of relevant work. In the controlled example below, the list must inspect every key for an absent lookup, while an explicitly constructed chain table inspects only the selected bucket.

The small hash function deliberately maps nonnegative integer IDs by remainder. This is a reproducible placement experiment, not instrumentation of .NET Dictionary.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

int[] ids = Enumerable.Range(0, 1000).ToArray();
int target = 1999;
int linearComparisons = 0;
foreach (int id in ids) { linearComparisons++; if (id == target) break; }
var buckets = new List<int>[100];
for (int i = 0; i < buckets.Length; i++) buckets[i] = new List<int>();
foreach (int id in ids) buckets[id % buckets.Length].Add(id);
int bucketComparisons = 0;
foreach (int id in buckets[target % buckets.Length])
{ bucketComparisons++; if (id == target) break; }
Console.WriteLine($"linear comparisons={linearComparisons}");
Console.WriteLine($"selected-chain comparisons={bucketComparisons}");

linear comparisons=1000
selected-chain comparisons=10


The absent lookup uses 1000 comparisons in the list and 10 in the selected chain. With a constant hash function, all keys would occupy one chain and the lookup would again require 1000 comparisons. Collision distribution explains the difference.

A benchmark should separate construction from the operation being studied, repeat lookups over representative hit/miss keys, warm up the runtime, report variation, and include realistic key-hashing costs. Those measurements complement the analysis; they do not turn expected constant-time hashing into a worst-case guarantee. Chapter 19 develops analysis and measurement further.

```{index} Dispatch from Two Consistent Indexes
```

## Dispatch from Two Consistent Indexes




```{index} priority queue; dictionary; consistency
```

The service desk keeps active records in a dictionary and pending IDs in a priority queue. A simple cancellation policy removes the active record while leaving its old queue entry in place. On dequeue, skip IDs no longer active. This is **lazy deletion**: defer physical cleanup until an obsolete entry reaches the front.

This example allows each ID to be registered once during the batch. If an ID can be reused or reprioritized, an old entry may still refer to an active ID. Use a generation/version field and verify it on dequeue, or implement an indexed heap with explicit updates. Presence alone would then be insufficient.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var active = new Dictionary<string, string>();
var registered = new HashSet<string>();
var pending = new PriorityQueue<string, (int Urgency, long Arrival)>();
long arrival = 0;
bool Register(string id, string owner, int urgency)
{
    if (!registered.Add(id)) return false;
    active.Add(id, owner);
    pending.Enqueue(id, (urgency, arrival++));
    return true;
}
Register("T-100", "Amina", 2);
Register("T-101", "Diego", 0);
Register("T-102", "Mina", 2);
Console.WriteLine($"duplicate accepted={Register("T-100", "Other", 0)}");
active.Remove("T-101");
while (pending.TryDequeue(out string? id, out _))
{
    if (!active.TryGetValue(id, out string? owner)) { Console.WriteLine($"skip cancelled {id}"); continue; }
    active.Remove(id);
    Console.WriteLine($"serve {id}: {owner}");
}
Console.WriteLine($"active={active.Count}, pending={pending.Count}");

duplicate accepted=False
skip cancelled T-101
serve T-100: Amina
serve T-102: Mina
active=0, pending=0


The cancelled urgent ticket is skipped, then T-100 and T-102 are served in their arrival order. Immediately after cancellation, the dictionary has two entries but the queue still has three. Equal Counts are not the invariant. Every active record has one live pending entry, while pending may also contain inactive IDs.

Skipping one stale entry costs one heap removal. Across a batch, each enqueued entry is removed at most once, but a single request for the next active ticket may skip many cancelled entries. Cancellation also leaves queue storage occupied until cleanup. Analyze both the whole workload and an individual call; do not promise that lazy deletion makes every dispatch O(log n).

```{index} Explain the Selection
```

## Explain the Selection




```{index} Explain the Selection
```


A dynamic array supports fast indexed access and sequential reports. Linked storage makes known-node updates convenient but does not make locating a node free. Stacks and queues restrict access order. Balanced ordered collections support ordered lookup; a heap supports priority selection; a hash collection supports equality-based access.

For cancellation, the dictionary identifies active records while pending heap entries can become stale. Lazy deletion skips stale entries when removing them. This simple example forbids ID reuse during the batch; reuse or reprioritization needs a version policy. Treat that policy as an extension after ordinary dictionary and set operations, not as a prerequisite for understanding hashing.


```{index} Practice: Skip a Cancelled Entry
```

## Practice: Skip a Cancelled Entry




```{index} Practice: Skip a Cancelled Entry
```


Dispatch only IDs still present in the active dictionary. Keep the ID-reuse restriction explicit.


In [ ]:
using System;
using System.Collections.Generic;

var active=new Dictionary<string,string>(StringComparer.Ordinal);
var waiting=new PriorityQueue<string,int>();
active.Add("T1","Avery");waiting.Enqueue("T1",1);
active.Add("T2","Jordan");waiting.Enqueue("T2",0);
active.Remove("T2");
while(waiting.TryDequeue(out string? id,out int priority))
    { /* TODO: remove an active record and print it. */ }


In [1]:
using System;
using System.Collections.Generic;

var active=new Dictionary<string,string>(StringComparer.Ordinal);
var waiting=new PriorityQueue<string,int>();
active.Add("T1","Avery");waiting.Enqueue("T1",1);
active.Add("T2","Jordan");waiting.Enqueue("T2",0);
active.Remove("T2");
while(waiting.TryDequeue(out string? id,out int priority))
    if(active.Remove(id,out string? owner))Console.WriteLine($"{id}: {owner}");


T1: Avery


```{rubric} Footnotes
```


[^1903_collection_selection-1]: Coordinated indexes are an applied extension. Ordinary dictionary/set use and one complete collision strategy are the core learning goals.


[^development-reading]: Further reading: [OpenDSA, Introduction to Hashing](https://opendsax.cs.vt.edu/OpenDSA/Books/Everything/html/HashIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
